In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake connection successful!")

In [ ]:
# Load the prepared churn dataset

df = session.table("ML_PROJECT.CHURN.CHURN_DATASET")

print("Dataset loaded successfully!")
print("Total rows:", df.count())

In [ ]:
# Show the dataset columns

print("Columns:")
print(df.columns)

In [ ]:
# Separate training and testing data

train_df = df.filter(df["DATA_SPLIT"] == "TRAIN")
test_df = df.filter(df["DATA_SPLIT"] == "TEST")

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

In [ ]:
# Define input features and target

feature_cols = [
    "SENIORCITIZEN",
    "TENURE",
    "MONTHLYCHARGES",
    "TOTALCHARGES",
    "GENDER_MALE",
    "PARTNER_YES",
    "DEPENDENTS_YES",
    "PHONE_SERVICE",
    "PAPERLESS_BILLING",
    "CONTRACT_MONTH_TO_MONTH",
    "CONTRACT_ONE_YEAR",
    "CONTRACT_TWO_YEAR",
    "INTERNET_DSL",
    "INTERNET_FIBER",
    "INTERNET_NONE"
]

target_col = "CHURN_TARGET"

print("Number of features:", len(feature_cols))
print("Target:", target_col)

In [ ]:
# Check churn distribution in training data

train_df.group_by("CHURN_TARGET").count().sort("CHURN_TARGET").show()

In [ ]:
# Test Snowflake ML

from snowflake.ml.modeling.ensemble import RandomForestClassifier

print("Snowflake ML imported successfully!")

In [ ]:
# Train the Random Forest model

model = RandomForestClassifier(
    input_cols=feature_cols,
    label_cols=[target_col],
    output_cols=["PREDICTION"],
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(train_df)

print("Model training completed successfully!")

In [ ]:
# Set the current database and schema

session.sql("USE DATABASE ML_PROJECT").collect()
session.sql("USE SCHEMA CHURN").collect()

print("Database:", session.sql("SELECT CURRENT_DATABASE()").collect()[0][0])
print("Schema:", session.sql("SELECT CURRENT_SCHEMA()").collect()[0][0])

In [ ]:
# Train the Random Forest model

model = RandomForestClassifier(
    input_cols=feature_cols,
    label_cols=[target_col],
    output_cols=["PREDICTION"],
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(train_df)

print("Model training completed successfully!")

In [ ]:
# Add packages required by Snowflake ML training

session.add_packages(
    "pandas",
    "numpy",
    "scikit-learn"
)

print("ML packages added successfully!")

In [ ]:
# Train the Random Forest model

model = RandomForestClassifier(
    input_cols=feature_cols,
    label_cols=[target_col],
    output_cols=["PREDICTION"],
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(train_df)

print("Model training completed successfully!")

In [ ]:
# Check Python ML packages available in the notebook

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Scikit-learn is available!")

In [ ]:
# Select only the ML columns and bring them into Python

train_data = train_df.select(
    feature_cols + [target_col]
).to_pandas()

test_data = test_df.select(
    feature_cols + [target_col]
).to_pandas()

print("Training data shape:", train_data.shape)
print("Testing data shape:", test_data.shape)

In [ ]:
# Separate input features and target

X_train = train_data[feature_cols]
y_train = train_data[target_col]

X_test = test_data[feature_cols]
y_test = test_data[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
# Train Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# Make predictions on the test data

y_pred = rf_model.predict(X_test)

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

In [ ]:
# Evaluate the model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Evaluation")
print("-----------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

In [ ]:
# Create confusion matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix")
print("----------------")
print(cm)

print("\nInterpretation:")
print("TN (True Negative):", cm[0][0])
print("FP (False Positive):", cm[0][1])
print("FN (False Negative):", cm[1][0])
print("TP (True Positive):", cm[1][1])

In [ ]:
# Prepare predictions for saving into Snowflake

prediction_data = test_data.copy()

prediction_data["PREDICTED_CHURN"] = y_pred

print("Prediction table prepared!")
print("Rows:", len(prediction_data))
print("Columns:", prediction_data.shape[1])

In [ ]:
# Save predictions back into Snowflake

session.write_pandas(
    prediction_data,
    table_name="CHURN_PREDICTIONS",
    database="ML_PROJECT",
    schema="CHURN",
    auto_create_table=True,
    overwrite=True
)

print("Predictions saved to Snowflake successfully!")

In [ ]:
%%sql -r dataframe_1
SELECT
    COUNT(*) AS TOTAL_PREDICTIONS,
    SUM(CASE WHEN PREDICTED_CHURN = 1 THEN 1 ELSE 0 END) AS PREDICTED_CHURN,
    SUM(CASE WHEN PREDICTED_CHURN = 0 THEN 1 ELSE 0 END) AS PREDICTED_NO_CHURN
FROM ML_PROJECT.CHURN.CHURN_PREDICTIONS;

In [ ]:
# Register the trained Random Forest model in Snowflake Model Registry

from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="ML_PROJECT",
    schema_name="CHURN"
)

model_ref = registry.log_model(
    rf_model,
    model_name="CUSTOMER_CHURN_RF",
    version_name="V1",
    sample_input_data=X_train.head(10),
    metrics={
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1)
    },
    conda_dependencies=["scikit-learn"],
    comment="Customer churn prediction Random Forest model"
)

print("Model registered successfully!")
print("Model: CUSTOMER_CHURN_RF")
print("Version: V1")

In [ ]:
registered_model = registry.get_model("CUSTOMER_CHURN_RF").version("V1")

registered_predictions = registered_model.run(
    X_test.head(10),
    function_name="predict"
)

print(registered_predictions)

In [ ]:
from snowflake.ml.model import target_platform

model_ref_v2 = registry.log_model(
    rf_model,
    model_name="CUSTOMER_CHURN_RF",
    version_name="V2",
    sample_input_data=X_train.head(10),
    metrics={
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1)
    },
    conda_dependencies=["scikit-learn"],
    target_platforms=target_platform.WAREHOUSE_ONLY,
    comment="Customer churn prediction Random Forest model - Warehouse inference"
)

print("Model V2 registered successfully!")
print("Model: CUSTOMER_CHURN_RF")
print("Version: V2")

In [ ]:
registered_model_v2 = registry.get_model(
    "CUSTOMER_CHURN_RF"
).version("V2")

registered_predictions = registered_model_v2.run(
    X_test.head(10),
    function_name="predict"
)

print("Inference successful!")
print(registered_predictions)

In [ ]:
feature_importance = pd.DataFrame({
    "FEATURE": feature_cols,
    "IMPORTANCE": rf_model.feature_importances_
}).sort_values(
    "IMPORTANCE",
    ascending=False
)

print(feature_importance)

In [ ]:
session.write_pandas(
    feature_importance,
    table_name="FEATURE_IMPORTANCE",
    database="ML_PROJECT",
    schema="CHURN",
    auto_create_table=True,
    overwrite=True
)

print("Feature importance saved to Snowflake successfully!")

In [ ]:
%%sql -r dataframe_2
SELECT *
FROM ML_PROJECT.CHURN.FEATURE_IMPORTANCE
ORDER BY IMPORTANCE DESC;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE VIEW ML_PROJECT.CHURN.CHURN_SUMMARY AS
SELECT
    COUNT(*) AS TOTAL_CUSTOMERS,
    SUM(CASE WHEN PREDICTED_CHURN = 1 THEN 1 ELSE 0 END) AS PREDICTED_CHURN,
    SUM(CASE WHEN PREDICTED_CHURN = 0 THEN 1 ELSE 0 END) AS PREDICTED_NO_CHURN
FROM ML_PROJECT.CHURN.CHURN_PREDICTIONS;